[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-1_%EC%84%BC%EC%84%9C%EC%8B%A0%ED%98%B8%EC%99%80_%ED%8A%B9%EC%A7%95%EC%B6%94%EC%B6%9C.ipynb)

# 2-1 · 센서 신호에서 특징(feature)을 뽑는다

**⏱ 60분** · 모듈 2: 센서 → **특징** → 이상탐지 → 고장진단

## 🎯 이번 시간의 질문

> 압력 센서 하나가 60초 동안 **6,000개**의 값을 기록합니다. 이 긴 신호로 부품 상태를 어떻게 비교할까요?

긴 신호를 그대로 비교하기는 어렵습니다. 그래서 신호를 **몇 개의 숫자(특징, feature)**로 요약합니다. 평균, 표준편차, 최댓값 같은 것들입니다.
특징을 잘 고르면 부품 상태에 따라 값이 뚜렷하게 달라지고, 이후의 이상탐지·진단이 쉬워집니다. **CBM에서 가장 먼저, 가장 많이 하는 작업**입니다(13일차 특성인자와 같은 내용).

## 데이터: 유압 시험장치 (독일 ZeMA 연구소, 공개 데이터)

유압 장치가 **60초짜리 같은 작업 사이클**을 2,205번 반복하는 동안, 연구자들이 네 부품의 상태를 일부러 바꿔 가며 센서를 기록했습니다.

| 부품 | 열 이름 | 상태 값 (정상 → 고장 직전) |
|---|---|---|
| 냉각기 | `cooler` | 100 (정상 효율) · 20 (효율 저하) · 3 (거의 고장) |
| 밸브 | `valve` | 100 (정상 전환) · 90 (약간 지연) · 80 (심한 지연) · 73 (거의 고장) |
| 펌프 | `pump_leakage` | 0 (누설 없음) · 1 (약한 누설) · 2 (심한 누설) |
| 축압기 | `accumulator` | 130 (정상 압력) · 115 · 100 · 90 (거의 고장) |

센서는 압력(PS, 100Hz), 모터 전력(EPS, 100Hz), 유량(FS, 10Hz), 온도(TS, 1Hz), 진동(VS, 1Hz) 등 17개입니다.
원신호는 그중 4개(PS1, FS1, TS1, VS1)를 제공하고, 나머지는 미리 계산한 특징 표로 제공합니다.

**이번 시간에 익힐 패턴:** 배열에서 요약값 계산(`mean`, `std`, `axis=1`) · P3 `groupby` · P4 선그래프·상자그림

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/hydraulic_features.csv",
             "data/hydraulic_raw_4sensors.npz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
labels = pd.read_csv('data/hydraulic_labels.csv')
raw = np.load('data/hydraulic_raw_4sensors.npz')
PS1, FS1, TS1, VS1 = raw['PS1'], raw['FS1'], raw['TS1'], raw['VS1']
print('읽기 완료')

## 1. 상태 표(정답) 보기

`labels`는 사이클마다 네 부품의 **실제 상태**가 적힌 표입니다. 나중에 진단 모델의 정답이 됩니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print(labels.shape)
labels.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 2,205행 = 사이클 2,205번. `stable_flag`는 조건을 바꾼 직후라 아직 안정되지 않았을 수 있으면 1입니다. `split`은 모델 학습용(train)과 평가용(test)을 미리 나눈 표시입니다(2-2부터 사용).

부품 상태별로 몇 사이클씩 있는지 `value_counts()`로 셉니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
labels['cooler'].value_counts()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


## 2. 원신호 들여다보기

원신호는 **2차원 배열**입니다. 행 = 사이클, 열 = 사이클 안의 시간 순서입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print('PS1:', PS1.shape)
print('FS1:', FS1.shape)
print('TS1:', TS1.shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 PS1은 (2205, 6000): 2,205개 사이클 × 6,000개 값입니다. 100Hz(1초에 100번) × 60초 = 6,000. FS1은 10Hz라서 600개, TS1은 1Hz라서 60개입니다.

**P4 · 선그래프** — 첫 번째 사이클(`PS1[0]`)의 압력을 시간 축으로 그립니다. 6,000개 값의 시간은 `0, 0.01, 0.02, … 59.99초`입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
t = np.arange(6000) / 100
plt.plot(t, PS1[0])
plt.xlabel('사이클 안의 시간 (초)')
plt.ylabel('압력 PS1 (bar)')
plt.title('사이클 1의 압력 신호')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 압력이 계단처럼 올라갔다 내려갑니다. 장치가 60초 동안 정해진 순서로 부하를 바꾸기 때문입니다. 모든 사이클이 같은 순서를 반복하므로, **같은 시점끼리 비교**하거나 **요약값끼리 비교**할 수 있습니다.

네 센서를 한꺼번에 봅니다.

In [ ]:
# ▶ 그대로 실행하세요
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, (name, sig, hz, unit) in zip(axes.flat, [('PS1 압력', PS1, 100, 'bar'), ('FS1 유량', FS1, 10, 'L/min'),
                                                   ('TS1 온도', TS1, 1, '°C'), ('VS1 진동', VS1, 1, 'mm/s')]):
    ax.plot(np.arange(sig.shape[1]) / hz, sig[0])
    ax.set_title(name); ax.set_xlabel('시간 (초)'); ax.set_ylabel(unit)
fig.tight_layout()
plt.show()

💬 유량은 압력과 비슷한 계단 모양, 온도는 한 사이클 안에서 거의 일정, 진동은 작은 폭으로 흔들립니다. 센서마다 **어떤 요약값이 의미 있을지**가 다릅니다.

## 3. 특징 만들기 — 6,000개 값을 숫자 몇 개로

한 사이클의 압력 신호를 네 숫자로 요약합니다.

| 특징 | 의미 |
|---|---|
| 평균 `mean()` | 신호의 전체 수준 |
| 표준편차 `std()` | 신호가 흔들리는 크기 |
| 최댓값 `max()`, 최솟값 `min()` | 극단값 |

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
x = PS1[0]
print('평균:', x.mean())
print('표준편차:', x.std())
print('최댓값:', x.max())
print('최솟값:', x.min())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 6,000개의 숫자가 4개의 숫자로 줄었습니다. 이제 사이클끼리 비교할 수 있습니다.

**모든 사이클을 한 번에** — `PS1.mean(axis=1)`은 **각 행(사이클)마다 가로 방향으로** 평균을 냅니다. 결과는 사이클 수만큼(2,205개)의 값입니다. 네 센서의 평균을 표로 만듭니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
feat = pd.DataFrame({'PS1_mean': PS1.mean(axis=1),
                     'FS1_mean': FS1.mean(axis=1),
                     'TS1_mean': TS1.mean(axis=1),
                     'VS1_mean': VS1.mean(axis=1)})
feat.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 `axis=1` = 가로 방향(한 사이클 안의 시간 방향). `axis=0`이었다면 세로 방향(같은 시각의 여러 사이클)으로 평균을 냈을 것입니다.

### ▶ 17개 센서 전체 특징 표

같은 방식으로 17개 센서 × 5개 특징(평균 `mean`, 표준편차 `std`, 최솟값 `min`, 최댓값 `max`, 마지막 1초 − 첫 1초 `change`)을 미리 계산해 둔 표를 읽고, 상태 표와 `cycle_id`로 합칩니다.
내가 계산한 값과 같은지도 확인합니다.

In [ ]:
# ▶ 그대로 실행하세요
features = pd.read_csv('data/hydraulic_features.csv')
data = labels.merge(features, on='cycle_id')
print('특징 표:', features.shape, '→ 합친 표:', data.shape)
print('내 계산과 같은가?', np.allclose(feat['PS1_mean'], data['PS1_mean']))

✅ `True`가 나오면 제공된 특징 표가 원신호에서 같은 방법으로 계산된 것입니다.

## 4. 좋은 특징 찾기 — 상태에 따라 값이 달라지는가

**P3 · 묶어 요약** — 냉각기 상태별로 오일 온도(`TS1_mean`) 평균을 비교합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data.groupby('cooler')['TS1_mean'].mean()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


**P4 · 상자그림** — 평균만 보면 흩어짐을 알 수 없습니다. 상자그림은 그룹마다 값의 **범위와 가운데**를 보여 줍니다. pandas 표에서 한 줄로 그릴 수 있습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data.boxplot(column='TS1_mean', by='cooler')
plt.suptitle('')          # pandas가 자동으로 붙이는 윗제목 지우기
plt.xlabel('냉각기 상태 (%)')
plt.ylabel('오일 온도 TS1 평균 (°C)')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 냉각기 효율이 떨어질수록(100 → 20 → 3) 오일 온도가 약 **36 → 45 → 55°C**로 올라가고, 세 상자가 거의 겹치지 않습니다.
냉각이 안 되면 기름이 뜨거워지는, **물리적으로 당연한 관계**가 데이터에 그대로 나타났습니다.
이렇게 **상태에 따라 값이 뚜렷이 갈리는 특징**이 좋은 특징입니다. `TS1_mean` 하나만으로도 냉각기 상태를 거의 알 수 있습니다.

상자 밖의 동그라미(이상치)는 무엇일까요? 냉각기 상태를 **바꾼 직후** 기름 온도가 아직 새 온도로 옮겨 가는 중인 사이클입니다(시험 시작 직후 1~31번, 상태를 바꾼 733번·1465번 직후 등, 모두 `stable_flag = 1`). 현장에서도 **정비 직후나 운전 조건을 바꾼 직후의 과도 구간**은 판단에서 따로 다뤄야 합니다.

---
## 5. 실습 과제 (25분)

### 📝 과제 1 · 펌프 누설을 잘 보여 주는 특징 찾기
펌프 누설(`pump_leakage` 0/1/2)에 따라 값이 가장 뚜렷이 달라지는 특징은 무엇일까요?
`FS1_mean`(유량), `PS1_mean`(압력), `EPS1_mean`(모터 전력), `VS1_mean`(진동) 중에서 비교하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 4절 코드에서 'cooler' → 'pump_leakage', 'TS1_mean' → 비교할 특징으로 바꿉니다.


✏️ **나의 답:** 어느 특징이 가장 좋아 보이나요? 냉각기-온도의 상자그림과 비교하면 어떤 점이 다른가요?

### 📝 과제 2 · 밸브 상태에 따라 파형이 어디서 달라지나?
아래 네 사이클은 **다른 부품은 모두 정상**이고 밸브 상태만 다릅니다: 사이클 1788(밸브 100), 1778(90), 1768(80), 1758(73).
네 사이클의 PS1 압력 신호를 한 그림에 겹쳐 그리고, **어느 시간대에서 차이가 나는지** 찾으세요.

⚠️ 배열의 행 번호는 0부터 시작하므로 사이클 1788은 `PS1[1787]`입니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: plt.plot(t, PS1[1787], label='밸브 100') 을 네 번 (label을 다르게) 부른 뒤 plt.legend()
# 힌트 2: 차이가 잘 안 보이면 plt.xlim(8, 12) 로 8~12초만 확대해 보세요.


✏️ **나의 답:** 어느 시간대에서, 어떻게 다른가요? 사이클 전체의 평균(`PS1_mean`)으로 이 차이를 잡을 수 있을까요?

### 📝 과제 3 · 새 특징 만들기 (도전)
과제 2에서 찾은 구간을 이용해 **"9~10초 구간의 압력 평균"** 특징을 모든 사이클에 대해 만들고, 밸브 상태별 평균을 비교하세요.
9~10초는 PS1 배열의 900번째~999번째 열입니다: `PS1[:, 900:1000]` (`:` = 모든 행)

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: data['PS1_peak9'] = PS1[:, 900:1000].mean(axis=1)  → 그다음 groupby('valve')


✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 원신호 | 2차원 배열 (사이클 × 시간). 100Hz × 60초 = 6,000개 |
| 특징 추출 | 평균·표준편차·최대·최소로 요약. `배열.mean(axis=1)`로 모든 사이클 한 번에 |
| 좋은 특징 | 상태에 따라 값이 **뚜렷이 갈리는** 특징. 냉각기-온도는 매우 좋고, 펌프-유량은 겹침이 크다 |
| 특징 설계 | 전체 평균에 안 보이는 차이가 **특정 구간**에 숨어 있을 수 있다 (밸브 9~10초) |

다음: **2-2 · 이상탐지 — 정상 데이터만으로 "평소와 다름"을 찾는다**